# Segment videos before the `EXPLANATION` frame

This notebook scans every video in `DATA/`, finds the first frame whose OCR text contains `EXPLANATION`, and saves the part before that frame into `SEGMENTED/` using exactly the same filename as the source video.

In [5]:
from pathlib import Path
import re
import shutil

import cv2
import numpy as np
from rapidocr_onnxruntime import RapidOCR
from tqdm.auto import tqdm

DATA_DIR = Path("DATA")
OUTPUT_DIR = Path("SEGMENTED")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_WORD = "EXPLANATION"
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".webm"}

# Keep this at 1 for the most accurate cut. Increase it only if scanning is too slow.
SCAN_EVERY_N_FRAMES = 1

COPY_WHOLE_VIDEO_IF_NOT_FOUND = False

ocr_engine = RapidOCR()

In [6]:
def preprocess_for_ocr(frame):
    """Prepare a video frame so title-card text is easier for OCR to read."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape

    # Upscale small videos because OCR is much more reliable on larger text.
    scale = 2 if max(height, width) < 1600 else 1
    if scale > 1:
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return thresh


def frame_has_target_word(frame, target_word=TARGET_WORD):
    image = preprocess_for_ocr(frame)
    result, _ = ocr_engine(image)
    lines = [] if result is None else [item[1] for item in result]
    text = " ".join(lines)
    normalized = re.sub(r"[^A-Z]", "", text.upper())
    return target_word in normalized, text.strip()


def find_first_target_frame(video_path, every_n_frames=SCAN_EVERY_N_FRAMES):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_index = 0

    with tqdm(total=total_frames, desc=f"Scanning {video_path.name}", unit="frame", leave=False) as pbar:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if frame_index % every_n_frames == 0:
                found, ocr_text = frame_has_target_word(frame)
                if found:
                    cap.release()
                    return frame_index, ocr_text

            frame_index += 1
            pbar.update(1)

    cap.release()
    return None, ""

In [7]:
def save_video_before_frame(video_path, output_path, stop_frame_index):
    """Write frames [0, stop_frame_index) to output_path."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not create output video: {output_path}")

    written = 0
    with tqdm(total=stop_frame_index, desc=f"Writing {output_path.name}", unit="frame", leave=False) as pbar:
        while written < stop_frame_index:
            ok, frame = cap.read()
            if not ok:
                break
            writer.write(frame)
            written += 1
            pbar.update(1)

    cap.release()
    writer.release()
    return written


def segment_video(video_path):
    output_path = OUTPUT_DIR / video_path.name
    target_frame, ocr_text = find_first_target_frame(video_path)

    if target_frame is None:
        if COPY_WHOLE_VIDEO_IF_NOT_FOUND:
            shutil.copy2(video_path, output_path)
            return {"video": video_path.name, "status": "copied_whole_video", "target_frame": None, "written_frames": None}
        return {"video": video_path.name, "status": "target_not_found", "target_frame": None, "written_frames": 0}

    written = save_video_before_frame(video_path, output_path, target_frame)
    return {
        "video": video_path.name,
        "status": "segmented",
        "target_frame": target_frame,
        "written_frames": written,
        "ocr_text": ocr_text,
    }

In [8]:
videos = sorted(path for path in DATA_DIR.iterdir() if path.suffix.lower() in VIDEO_EXTENSIONS)
print(f"Found {len(videos)} video(s) in {DATA_DIR.resolve()}")

results = []
for video_path in tqdm(videos, desc="Processing videos", unit="video"):
    try:
        result = segment_video(video_path)
    except Exception as exc:
        result = {"video": video_path.name, "status": "error", "error": str(exc)}
    results.append(result)
    print(result)

print("Done. Segmented videos are in:", OUTPUT_DIR.resolve())

Found 259 video(s) in F:\TCSR\Seg\DATA


Processing videos:   0%|          | 1/259 [02:16<9:46:44, 136.45s/video]

{'video': 'Ability.mp4', 'status': 'segmented', 'target_frame': 161, 'written_frames': 161, 'ocr_text': 'FINSERV BAJAJ Explanation of Ability'}


Processing videos:   1%|          | 2/259 [04:44<10:13:49, 143.30s/video]

{'video': 'Acknowledge.mp4', 'status': 'segmented', 'target_frame': 164, 'written_frames': 164, 'ocr_text': 'FINSERV BAJAJ Explanation of Acknowledge'}


...

{'video': 'Symptoms.mp4', 'status': 'segmented', 'target_frame': 155, 'written_frames': 155, 'ocr_text': 'FINSERV BAJAJ Explanation of Symptoms'}


Processing videos:  94%|█████████▍| 244/259 [23:25:38<47:07, 188.51s/video]

{'video': 'Tantrums.mp4', 'status': 'segmented', 'target_frame': 142, 'written_frames': 142, 'ocr_text': 'FINSERV BaJJ Explanation of Tantrums'}


Processing videos:  94%|█████████▍| 244/259 [23:28:13<1:26:34, 346.28s/video]


KeyboardInterrupt: 